# RAPID response prediction: expanded grouped-CV grid

This version locates the repository automatically, so it also works when Jupyter starts in `RAPID/notebooks` or another subdirectory.

## Candidate predictors

Resistance / footprint: `len_fraction_width_weighted_log_lambda`, `length_fraction`, `width_fraction`, `k_ratio_to_matched_base`, `along_network_frac`.

Geometry / topology: `equivalent_length`, `elongation`, `topologic_complexity_score`, `effective_n_paths_width`, `dominant_width_fraction`, `path_length_cv`, `branching_density_by_length`.

Independent-collapse topology values come from the matching base-state unit. Compound-bubble values are equivalent-width-weighted means of member-unit values.

The search verifies before fitting that `width_fraction` is the shared domain-aligned exposure factor and that `len_fraction_width_weighted_log_lambda` equals `length_fraction × -log10(Lambda_matched) × width_fraction`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'RAPID/classification/search_rapid_response_predictors.py').is_file():
            return candidate
    raise FileNotFoundError('Could not find river-hierarchy repo. Set REPO_ROOT manually.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_ROOT = Path('/Volumes/PhD/river_hierarchy/network_variants/output')
EXAMPLE_IDS = ['02', '03', '05', '11', '13','15', '21','22','37','41','45','46','48','54', '55', '56']
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = REPO_ROOT / 'RAPID/classification/results' / f'rapid-response-grid-v1-{STAMP}'

assert OUTPUT_ROOT.exists(), f'Missing RAPID output root: {OUTPUT_ROOT}'
print('Repository root:', REPO_ROOT)
RUN_DIR
RAPID_SRC = REPO_ROOT / 'RAPID/src'
if str(RAPID_SRC) not in sys.path:
    sys.path.insert(0, str(RAPID_SRC))
from rapid_tools.response_diagnostics import response_scores

Repository root: /Users/6256481/Code/river-hierarchy


## Run the search

This evaluates 794 predictor sets (all one- through four-variable combinations, plus the full 12-variable set), 12 model settings, eight leave-one-example-out folds, and two separate response targets. It can take a substantial time.

In [2]:
command = [
    sys.executable, str(REPO_ROOT / 'RAPID/classification/search_rapid_response_predictors.py'),
    '--output-root', str(OUTPUT_ROOT), '--results-dir', str(RUN_DIR),
    '--example-ids', *EXAMPLE_IDS,
]
completed = subprocess.run(command, cwd=REPO_ROOT, check=True, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

Wrote /Users/6256481/Code/river-hierarchy/RAPID/classification/results/rapid-response-grid-v1-20260921T103432Z
Best lag_to_inflow_peak_pct_shift_vs_base: ridge_alpha_10 | len_fraction_width_weighted_log_lambda + length_fraction + elongation + branching_density_by_length | MAE=0.1281
Best peak_discharge_pct_shift_vs_base: tree_d2_leaf8 | length_fraction | MAE=0.0307



## Compare screening candidates

Lower MAE is better. These are screening rankings, rather than final unbiased scores: use physically interpretable, similarly scoring candidates as the preferred model, and validate on new examples.

In [4]:
scores = pd.read_csv(RUN_DIR / 'grouped_cv_grid_scores.csv').sort_values(
    ['r2', 'target', 'mae_pct_points', 'n_features', 'rmse_pct_points'],
    ascending=[False, True, True, True, True])
# New runs write all score scales. The next diagnostics also support archived predictions.
for target, table in scores.groupby('target', sort=False):
    print(f'\n{target}')
    columns = ['model', 'features', 'mae_pct_points', 'r2', 'r2_signed_log1p',
               'r2_log10_magnitude', 'n_log_magnitude', 'sign_accuracy', 'n_features']
    display(table[[c for c in columns if c in table]].head(15).reset_index(drop=True))


lag_to_inflow_peak_pct_shift_vs_base


,model,features,mae_pct_points,r2,r2_signed_log1p,r2_log10_magnitude,n_log_magnitude,sign_accuracy,n_features
0,ridge_alpha_0.01,len_fraction_width_weighted_log_lambda + lengt...,0.130980,0.821071,0.666773,0.323314,328,0.765903,4
1,ridge_alpha_0.1,len_fraction_width_weighted_log_lambda + lengt...,0.130937,0.821065,0.666888,0.311873,328,0.765903,4
2,ridge_alpha_1,len_fraction_width_weighted_log_lambda + lengt...,0.130533,0.820975,0.667930,0.312663,328,0.768448,4
3,ridge_alpha_0.01,len_fraction_width_weighted_log_lambda + lengt...,0.130488,0.820958,0.668940,0.281364,328,0.811705,4
4,ridge_alpha_0.1,len_fraction_width_weighted_log_lambda + lengt...,0.130448,0.820953,0.669041,0.296101,328,0.811705,4
5,ridge_alpha_1,len_fraction_width_weighted_log_lambda + lengt...,0.130078,0.820876,0.669971,0.355714,328,0.811705,4
6,ridge_alpha_0.01,length_fraction + k_ratio_to_matched_base + el...,0.138612,0.820714,0.654165,0.209258,328,0.740458,4
7,ridge_alpha_0.1,length_fraction + k_ratio_to_matched_base + el...,0.138596,0.820700,0.654254,0.216247,328,0.740458,4
8,ridge_alpha_1,length_fraction + k_ratio_to_matched_base + el...,0.138431,0.820550,0.655128,0.254859,328,0.740458,4
9,ridge_alpha_0.01,len_fraction_width_weighted_log_lambda + along...,0.129830,0.820515,0.668202,0.236054,328,0.786260,4



peak_discharge_pct_shift_vs_base


,model,features,mae_pct_points,r2,r2_signed_log1p,r2_log10_magnitude,n_log_magnitude,sign_accuracy,n_features
0,tree_d2_leaf8,length_fraction,0.030712,0.406679,0.391466,-0.149936,393,0.92112,1
1,tree_d2_leaf8,len_fraction_width_weighted_log_lambda + lengt...,0.030712,0.406679,0.391466,-0.149936,393,0.92112,2
2,tree_d2_leaf8,along_network_frac + length_fraction,0.030712,0.406679,0.391466,-0.149936,393,0.92112,2
3,tree_d2_leaf8,length_fraction + width_fraction,0.030712,0.406679,0.391466,-0.149936,393,0.92112,2
4,tree_d2_leaf8,length_fraction + k_ratio_to_matched_base,0.030712,0.406679,0.391466,-0.149936,393,0.92112,2
5,tree_d2_leaf8,length_fraction + elongation,0.030712,0.406679,0.391466,-0.149936,393,0.92112,2
6,tree_d2_leaf8,length_fraction + topologic_complexity_score,0.030712,0.406679,0.391466,-0.149936,393,0.92112,2
7,tree_d2_leaf8,length_fraction + effective_n_paths_width,0.030712,0.406679,0.391466,-0.149936,393,0.92112,2
8,tree_d2_leaf8,length_fraction + dominant_width_fraction,0.030712,0.406679,0.391466,-0.149936,393,0.92112,2
9,tree_d2_leaf8,length_fraction + path_length_cv,0.030712,0.406679,0.391466,-0.149936,393,0.92112,2


In [5]:
cohort = pd.read_csv(RUN_DIR / 'common_complete_case_cohort.csv')
provenance = pd.read_csv(RUN_DIR / 'topology_metric_provenance.csv')
exposure_audit = pd.read_csv(RUN_DIR / 'exposure_metric_audit.csv')
print(f'Complete-case states: {len(cohort)} across {cohort.example_id.nunique()} examples')
display(cohort.groupby(['example_id', 'mode']).size().rename('states').reset_index())
display(provenance)
print('Domain-aligned exposure audit (verified before model fitting):')
display(exposure_audit)

Complete-case states: 393 across 16 examples


,example_id,mode,states
0,2,indep,5
1,3,compound,1
2,3,indep,13
3,5,compound,1
4,5,indep,26
5,11,indep,3
6,13,compound,2
7,13,indep,26
8,15,compound,1
9,15,indep,22


,experiment_dir,independent_source,compound_source
0,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean
1,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean
2,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean
3,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean
4,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean
5,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean
6,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean
7,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean
8,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean
9,/Volumes/PhD/river_hierarchy/network_variants/...,direct_base_unit,equivalent_width_weighted_member_unit_mean


Domain-aligned exposure audit (verified before model fitting):


,exposure_fraction_source,states,raw_width_share_mean,exposure_factor_mean,states_changed_from_raw
0,complete_closed_compound,17,1.000000,1.000000,0
1,nested_or_unit_width_share,376,0.792831,0.792831,0


## Observed versus held-out predictions

Every point is predicted by a model that did not train on that example.

In [3]:
predictions = pd.read_csv(RUN_DIR / 'grouped_cv_grid_predictions.csv')
best = scores.groupby('target', as_index=False).first()
fig, axes = plt.subplots(1, len(best), figsize=(7 * len(best), 5), squeeze=False)
for axis, row in zip(axes.ravel(), best.itertuples(index=False)):
    plot_data = predictions[(predictions.target == row.target) & (predictions.model == row.model) & (predictions.features == row.features)]
    axis.scatter(plot_data.observed, plot_data.predicted, alpha=0.75)
    limits = [min(plot_data.observed.min(), plot_data.predicted.min()), max(plot_data.observed.max(), plot_data.predicted.max())]
    axis.plot(limits, limits, '--', color='black', linewidth=1)
    axis.set(xlabel='Observed response (% change)', ylabel='Held-out prediction (% change)', title=f'{row.target}\nLinear R² {row.r2:.3f}; MAE {row.mae_pct_points:.3f}; {row.model}')
plt.tight_layout()

NameError: name 'scores' is not defined

## Score scales and signed magnitude diagnostics

The primary grid still fits the signed percentage response. **Changing a plot axis does not change the fitting objective or R².** Every new grid/fixed-model run now saves:

- `r2_linear` (also `r2` for compatibility): ordinary signed-response R², sensitive to large absolute errors.
- `r2_signed_log1p`: R² after T(y) = sign(y) × ln(1 + |y| / s), with s = 1 percentage point. This retains signs and zeros, compresses large values, and depends on the stated scale. It is not an ordinary logarithm.
- `r2_log10_magnitude`: R² between log10(|observed|) and log10(|predicted|) for nonzero pairs. This measures multiplicative magnitude accuracy and **does not penalise the wrong sign**. Always read it alongside `sign_accuracy`, `n_log_magnitude`, and `n_zero_pairs_excluded`.
- `r2_linear_log_subset`: linear R² on the exact same nonzero subset, to separate transformation effects from population changes.

For each scale, R² = 1 − Σ(z − ẑ)² / Σ(z − mean(z))², using held-out predictions. Scores on different transformed targets have different denominators; their difference is not a gain in predictive skill on one common target. Re-scoring the same predictions tests evaluation-scale sensitivity; fitting a log target is a separate model choice, investigated by the conditional curves in the hydrograph notebook.

The following figures retain the original signs and exact-zero display conventions. The all-state signed grid and the conditional response-curve comparison answer related but different questions.

In [ ]:
from rapid_tools.response_diagnostics import make_signed_magnitude_scatter, show_signed_scatter
TIMING_TARGET = 'lag_to_inflow_peak_pct_shift_vs_base'
PRIMARY_PREDICTOR = 'len_fraction_width_weighted_log_lambda'
for log_x in (False, True):
    diagnostic = make_signed_magnitude_scatter(
        cohort, x=PRIMARY_PREDICTOR, y=TIMING_TARGET, log_x=log_x,
        title='Prediction cohort: resistance–timing magnitudes')
    show_signed_scatter(diagnostic)

# Re-score existing held-out predictions too; no broad grid rerun is needed for this comparison.
score_scale_rows = []
for keys, part in predictions.groupby(['target', 'model', 'features'], sort=False):
    score_scale_rows.append(dict(zip(['target', 'model', 'features'], keys)) |
                            response_scores(part.observed, part.predicted))
score_scale_comparison = pd.DataFrame(score_scale_rows)
for target, table in score_scale_comparison.groupby('target'):
    print(target)
    display(table.sort_values('r2_linear', ascending=False).head(15))

## Fitting on transformed responses

The score comparison above keeps each model's held-out predictions fixed. The hydrograph notebook separately fits power-law, exponential, and quadratic-in-log-space magnitude models with leave-one-example-out validation within each sign quadrant. A future signed conditional model would need to predict sign/zero status as well as magnitude. The reduced interpretation below distinguishes hydraulic resistance from routing travel time.

In [ ]:
# Directly quantify the evaluation-scale effect for the linearly best screened candidates.
scale_effect = best[['target', 'model', 'features']].merge(
    score_scale_comparison, on=['target', 'model', 'features'], validate='one_to_one')
display(scale_effect)

## Quick preliminary result: one fixed Ridge model

Use this section after a kernel restart when you need results for a selected example set without running the broad grid. It runs **only** the model selected for its strong grouped-CV R²:

`Ridge(alpha=10)` with `len_fraction_width_weighted_log_lambda`, `equivalent_length`, and `topologic_complexity_score`.

This is a preliminary, fixed-model evaluation—not fresh model selection. It retains the shared domain-aligned exposure verification and uses leave-one-example-out validation.

In [ ]:
# Change this list to the RAPID examples you want in the preliminary result.
PRELIM_EXAMPLE_IDS = ['02', '03', '05', '11', '13','15', '21','22','37','41','45','46','48','54', '55', '56']
PRELIM_STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
PRELIM_RUN_DIR = REPO_ROOT / 'RAPID/classification/results' / f'rapid-response-preliminary-ridge-v1-{PRELIM_STAMP}'

preliminary_command = [
    sys.executable, str(REPO_ROOT / 'RAPID/classification/run_rapid_response_preliminary_model.py'),
    '--output-root', str(OUTPUT_ROOT), '--results-dir', str(PRELIM_RUN_DIR),
    '--example-ids', *PRELIM_EXAMPLE_IDS,
]
try:
    completed = subprocess.run(preliminary_command, cwd=REPO_ROOT, check=True, text=True, capture_output=True)
except subprocess.CalledProcessError as error:
    print(error.stdout or '')
    print(error.stderr or '')
    raise
else:
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)

Wrote /Users/6256481/Code/river-hierarchy/RAPID/classification/results/rapid-response-preliminary-ridge-v1-20260903T055747Z
         model                                                                                features  n_features  mae_pct_points  rmse_pct_points       r2
ridge_alpha_10 len_fraction_width_weighted_log_lambda + equivalent_length + topologic_complexity_score           3        0.170966         0.287829 0.693274



In [ ]:
preliminary_score = pd.read_csv(PRELIM_RUN_DIR / 'grouped_cv_score.csv')
preliminary_predictions = pd.read_csv(PRELIM_RUN_DIR / 'grouped_cv_predictions.csv')
observed = preliminary_predictions['observed_lag_shift_pct'].to_numpy()
predicted = preliminary_predictions['predicted_lag_shift_pct'].to_numpy()
preliminary_scale_scores = response_scores(observed, predicted)
display(pd.DataFrame([preliminary_scale_scores]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(observed, predicted, alpha=0.65)
lo, hi = min(observed.min(), predicted.min()), max(observed.max(), predicted.max())
axes[0].plot([lo, hi], [lo, hi], '--', color='black')
axes[0].set(xlabel='RAPID lag shift [%]', ylabel='Held-out geometry prediction [%]',
            title=f"Signed response: linear R² = {preliminary_scale_scores['r2_linear']:.3f}")
valid = np.isfinite(observed) & np.isfinite(predicted) & (observed != 0) & (predicted != 0)
same_sign = np.sign(observed) == np.sign(predicted)
for correct, marker, label in [(True, 'o', 'Correct sign'), (False, 'D', 'Wrong sign')]:
    mask = valid & (same_sign == correct)
    axes[1].scatter(abs(observed[mask]), abs(predicted[mask]), marker=marker, label=label, alpha=0.65)
if valid.any():
    lo = min(abs(observed[valid]).min(), abs(predicted[valid]).min())
    hi = max(abs(observed[valid]).max(), abs(predicted[valid]).max())
    axes[1].plot([lo, hi], [lo, hi], '--', color='black')
axes[1].set(xscale='log', yscale='log', xlabel='|RAPID lag shift| [%]', ylabel='|Held-out prediction| [%]',
            title=f"Log10-magnitude R² = {preliminary_scale_scores['r2_log10_magnitude']:.3f}\n"
                  f"{preliminary_scale_scores['n_log_magnitude']} nonzero pairs; zeros excluded")
axes[1].legend()
plt.tight_layout()

## What does the width dependence actually imply?

The implemented geometry is $d=W/k_b$, $A=W^2/k_b$, and (wide-channel approximation) $R_h=W/k_b$. Substitution into the edge-resistance equations gives

$$K_{\mathrm{Manning}}=\frac{n^2\ell}{A^2R_h^{4/3}}
=n^2\ell\,k_b^{10/3}W^{-16/3},$$
$$K_{\mathrm{Chezy}}=\frac{\ell}{A^2C_z^2gR_h}
=\frac{\ell k_b^3}{C_z^2g}W^{-5}.$$

So **inverse width to the fifth power is the Chezy edge-resistance law**. All 32 manifests for the 16 configured examples (independent and compound) specify **Chezy, wide geometry**, so that inverse-fifth-power edge law applies to this data. Manning, available as an alternative, gives inverse width to power $16/3\approx5.33$. For rectangular geometry and fixed $k_b$, the width exponents remain the same but the coefficients change. These statements hold with length, roughness, and width-depth ratio fixed. They are not exponents for outlet timing or for a multi-edge resistance ratio when topology and geometry change together.

RAPID's uncapped routing celerity is separately computed as

$$c=\frac{5}{3n}\sqrt{S_0}\left(\frac{W}{k_b}\right)^{2/3},\qquad k_{\mathrm{routing}}=\ell_{\mathrm{effective}}/c.$$

Thus local celerity scales as $W^{2/3}$ under fixed slope, roughness and $k_b$, before any celerity caps; RAPID routing $k$ is not the hydraulic resistance $K$. Branch exposure also depends on *ratios* of widths, while complete-compound exposure is fixed at one. If every width were multiplied by the same factor in both comparison states, with the other parameters fixed, that common resistance factor would cancel in $\Lambda_K$ and the branch-width fractions would not change.

**Most reduced defensible takeaway:** this analysis tests whether relative resistance reduction, weighted by affected network length and flow exposure, predicts relative peak timing. It does not establish “timing is width to power five.” Width has a strong local influence on resistance, but network routing and topology determine how it appears in outlet timing. The held-out results below determine how predictive the chosen summary is.

Source equations: `hierarchy_level_definition/metrics/equivalent_resistance.py::compute_geometry`, `compute_K_manning`, `compute_K_chezy`; `RAPID/src/rapid_tools/k_values.py`. Check each experiment's resistance configuration before assigning it either law.

## Recorded score-scale audit of the existing fixed Ridge run

Source: `rapid-response-preliminary-ridge-v1-20260903T055747Z`, 393 held-out states from 16 examples. This reuses saved predictions, so no new training or full-grid search is implied. Linear R² = **0.6933**; signed-log1p R² (s = 1 percentage point) = **0.5292**. Log10-magnitude R² = **0.2196** on 328 nonzero pairs, versus linear R² = **0.6974** on those same pairs. There are 65 pairs excluded from the ordinary logarithm because one value is zero. Sign accuracy, including exact zeros, is **79.1%**.

The model explains large absolute timing changes better than it captures multiplicative differences among small nonzero changes. A log axis alone would conceal this scoring distinction. These numbers concern the archived fixed model; the grid table above computes each candidate's own values when run.

The archived prediction cohort also requires `boundary_relation == "exact_match"`; the primary hydrograph plot uses its documented finite-pair/edge-compound selection. Consequently their samples differ, and their R² values should not be treated as a head-to-head model comparison.


In [ ]:
# Reproduce the recorded audit without running the model-selection grid.
ARCHIVED_RIDGE_DIR = REPO_ROOT / 'RAPID/classification/results/rapid-response-preliminary-ridge-v1-20260903T055747Z'
archived_predictions = pd.read_csv(ARCHIVED_RIDGE_DIR / 'grouped_cv_predictions.csv')
archived_scale_scores = response_scores(archived_predictions.observed_lag_shift_pct,
                                       archived_predictions.predicted_lag_shift_pct)
display(pd.DataFrame([archived_scale_scores]))


,r2,r2_linear,r2_signed_log1p,signed_log_scale,r2_log10_magnitude,r2_linear_log_subset,n_scored,n_nonfinite_excluded,n_log_magnitude,n_zero_pairs_excluded,sign_accuracy
0,0.693274,0.693274,0.529227,1.0,0.219617,0.697425,393,0,328,65,0.791349
